# 💳 Credit Card Fraud Detection using Deep Neural Network

## 📌 Overview
Credit card fraud detection is a mission-critical challenge in financial systems characterized by **extreme class imbalance** (typically < 0.2% fraud) and high-velocity transactional data.

This project builds an end-to-end Deep Learning system:
1. **Exploratory Data Analysis (EDA)** on transaction volume and PCA features
2. **Class Imbalance Handling** via stratified batching and weighted loss functions
3. **Deep Neural Network (DNN)** with Batch Normalization and Dropout
4. **Unsupervised Deep Autoencoder** for anomaly reconstruction scoring
5. Evaluation on **PR-AUC, ROC-AUC, and Confusion Matrix**

In [ ]:
# 1. Environment & Library Setup
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn

from src.data_loader import load_and_preprocess_data, generate_benchmark_dataset
from src.models import FraudDNN, FraudAutoencoder
from src.train import train_dnn
from src.predict import FraudPredictor

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch active compute device: {device}')

## 📊 2. Dataset Loading & Inspection
We utilize the standard benchmark containing 28 PCA-transformed numerical features (`V1` to `V28`), `Time`, `Amount`, and the ground truth `Class` (0 = Legitimate, 1 = Fraud).

In [ ]:
data_path = 'data/creditcard.csv'
if not os.path.exists(data_path):
    generate_benchmark_dataset(data_path)

df = pd.read_csv(data_path)
print(f'Total records: {len(df):,}')
print(f'Features: {df.shape[1]}')
print('\nClass distribution:')
print(df['Class'].value_counts())
df.head()

## 🔍 3. Exploratory Data Analysis & Visualizations
Inspecting class imbalance, distribution of transaction amounts, and correlation.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Class Distribution
sns.countplot(x='Class', data=df, ax=axes[0], palette=['#1f77b4', '#d62728'])
axes[0].set_title('Transaction Class Imbalance', fontsize=12, fontweight='bold')
axes[0].set_yscale('log')
axes[0].set_ylabel('Log Count')

# Amount Distribution
sns.boxplot(x='Class', y='Amount', data=df, ax=axes[1], palette=['#1f77b4', '#d62728'], showfliers=False)
axes[1].set_title('Transaction Amount by Class', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

## 🧠 4. Deep Neural Network Architecture
Our model is a Multi-Layer Perceptron (MLP) Deep Neural Network featuring:
- Batch Normalization across hidden layers
- LeakyReLU activations with negative slope 0.1
- Dropout (rate 0.3) to prevent overfitting on minority patterns

In [ ]:
dnn_model = FraudDNN(input_dim=30).to(device)
print(dnn_model)

## ⚡ 5. Model Training
We train the model using AdamW optimizer and weighted BCE loss to handle the 0.17% fraud prevalence.

In [ ]:
metrics = train_dnn(epochs=12, batch_size=256, lr=0.001, data_path=data_path)

## 📈 6. Evaluation & Diagnostic Curves
Displaying Confusion Matrix and Precision-Recall / ROC curves.

In [ ]:
from IPython.display import Image, display
if os.path.exists('reports/figures/roc_pr_curves.png'):
    display(Image('reports/figures/roc_pr_curves.png'))
if os.path.exists('reports/figures/confusion_matrix.png'):
    display(Image('reports/figures/confusion_matrix.png'))
if os.path.exists('reports/figures/training_history.png'):
    display(Image('reports/figures/training_history.png'))

## 🚀 7. Real-time Inference Pipeline
Testing real-time scoring with the `FraudPredictor` inference engine.

In [ ]:
predictor = FraudPredictor()

# Sample legitimate transaction
legit_tx = {'Time': 1000.0, 'Amount': 34.50}
for i in range(1, 29): legit_tx[f'V{i}'] = 0.05

# Sample fraudulent transaction (elevated negative V14 and positive V4)
fraud_tx = {'Time': 50000.0, 'Amount': 850.00}
for i in range(1, 29): fraud_tx[f'V{i}'] = 0.0
fraud_tx['V14'] = -4.2
fraud_tx['V4'] = 3.6
fraud_tx['V11'] = 2.9

print('Legitimate Transaction Assessment:', predictor.predict(legit_tx))
print('Fraudulent Transaction Assessment:', predictor.predict(fraud_tx))